In [26]:
import numpy as np
import pandas as pd
import re, datetime
from word2number import w2n

# Functions
def fix_quantity(x) -> int:
    if isinstance(x, str):
        x_clean = x.strip().lower()
        try:
            # try converting word numbers
            return w2n.word_to_num(x_clean)
        except:
            try:
                # try converting numeric strings
                return int(x_clean)
            except:
                return x # return as is if failed conversion
    return x

def try_parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    iso_pattern = r"^\d{4}[-/]\d{2}[-/]\d{2}$"     # 2024-07-05 or 2024/07/05
    us_pattern = r"^\d{2}/\d{2}/\d{4}$"            # 07/05/2024

    try:
        if re.match(iso_pattern, date_str):
            return pd.to_datetime(date_str, errors='coerce', dayfirst=False)
        elif re.match(us_pattern, date_str):
            return pd.to_datetime(date_str, errors='coerce', dayfirst=False)
        else:
            return pd.to_datetime(date_str, errors='coerce', dayfirst=True)
    except:
        return pd.NaT

def convert_time(t):
    t = str(t).strip().lower()

    # Already labeled
    if t in ['morning', 'afternoon', 'evening']:
        return t

    # Try to parse to time
    try:
        parsed_time = pd.to_datetime(t).time()
        if datetime.time(5, 0) <= parsed_time < datetime.time(12, 0):
            return 'morning'
        elif datetime.time(12, 0) <= parsed_time < datetime.time(17, 0):
            return 'afternoon'
        elif datetime.time(17, 0) <= parsed_time < datetime.time(21, 0):
            return 'evening'
        else:
            return 'night'  # Optional
    except:
        return 'unknown'

# Setup
df=pd.read_csv('park_sightings.csv')

# ObserverName
df['ObserverName']=df['ObserverName'].str.title()

# Species

# Count
df['Count']=df['Count'].apply(fix_quantity)

# Location
df['Location']=df['Location'].str.replace('Lakeview Tral', 'Lakeview Trail')

# DateObserved
df['DateObserved']=df['DateObserved'].apply(try_parse_date)

# TimeObserved
df['TimeObserved']=df['TimeObserved'].apply(convert_time)
df=pd.get_dummies(df, columns=['TimeObserved'], drop_first=True)

# Weather
df['Weather']=df['Weather'].str.lower()
df['Weather']=df['Weather'].str.replace(r'\bsun\b', 'sunny', regex=True)
df['Weather']=df['Weather'].map({
    'sunny': 0,
    'cloudy': 1,
    'overcast': 2,
    'rainy': 3
})

# Activity
df=pd.get_dummies(df, columns=['Activity'], drop_first=True)

# PhotoTaken
df['PhotoTaken']=df['PhotoTaken'].str.lower()
df['PhotoTaken']=df['PhotoTaken'].map({
    'n': 0, 'no': 0,
    'y': 1, 'yes': 1
})

# Others
df=df.drop(columns=['SightingID', 'Notes'])

# Output
# Force Display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df
# # df.isna().sum()
# for col in df.columns:
#     print(f"\n{col} ➤ Unique values:")
#     print(df[col].dropna().unique())


ObserverName ➤ Unique values:
['J.K. Chang' 'Chris Wu' 'Kim Lee' 'Maria Flores' 'Alex Taylor'
 'Dana Lopez' 'Ashley Moore' 'Jordan K.']

Species ➤ Unique values:
['Turtle' 'Squirrel' 'Deer' 'Hawk' 'Raccoon']

Count ➤ Unique values:
[ 9  8  1  7 10  4  5  3  6  2]

Location ➤ Unique values:
['North Meadow' 'Lakeview Trail' 'Central Path' 'Riverbend'
 'Woodland Edge']

Latitude ➤ Unique values:
[35.94256  35.747132 35.989993 35.839801 35.135813 35.37848  35.772681
 35.944747 35.094675 35.578943 35.631899 35.573388 35.474823 35.427756
 35.338478 35.358119 35.060836 35.031146 35.176423 35.855752 35.856095
 35.586306 35.313716 35.47299  35.367162 35.645422 35.862809 35.803059
 35.637846 35.211155 35.722316 35.473578 35.335755 35.622635 35.051557
 35.702044 35.526039 35.420044 35.601065 35.980445 35.253581 35.908495
 35.387073 35.251818 35.578586 35.883271 35.219264 35.968698 35.601914
 35.354347 35.205647 35.757768 35.702692 35.909433 35.436055 35.659826
 35.435829 35.780623 35.696613 35.4